# OpenAI Just Introduced Function Callings Feature: Everything You Need to Know

[Article](https://todatabeyond.com/blog/openai-just-introduced-function-callings-feature-everything-you-need-to-know). Original examples are preserved in the nonexecuting appendix below.

Requires an `OPENAI_API_KEY` in your environment; calls incur charges. `MODEL` defaults to `gpt-5.4-mini`, the tested model. Do not place credentials in cells.

Install: `pip install openai==2.43.0 jsonschema==4.26.0 python-dotenv==1.2.2`.

In [ ]:
import os, json, sys
from importlib.metadata import version
from openai import OpenAI
from jsonschema import validate, ValidationError
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=45)
def chat(messages, **kwargs):
    return client.chat.completions.create(
        model=MODEL, messages=messages, max_completion_tokens=1024,
        reasoning_effort="none", **kwargs)
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "jsonschema:", version("jsonschema"))
print("Model:", MODEL)

In [ ]:
weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string", "description": "City and state, for example Boston, MA"},
        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
    },
    "required": ["location", "unit"], "additionalProperties": False
}
tools = [{"type": "function", "function": {
    "name": "get_current_weather", "description": "Get weather for the requested city. Use fahrenheit unless the user asks for celsius.",
    "strict": True, "parameters": weather_schema
}}]
print(json.dumps(tools, indent=2))
messages = [{"role": "developer", "content": "Demonstrate a weather tool using a fixed teaching fixture. When given a tool result, summarize its temperature, unit and forecast. Call it an example, not live weather."},
            {"role": "user", "content": "What is the weather like in Boston, MA in celsius?"}]
response = chat(messages, tools=tools,
                tool_choice={"type": "function", "function": {"name": "get_current_weather"}},
                parallel_tool_calls=False)
assistant = response.choices[0].message
assert len(assistant.tool_calls or []) == 1
print(assistant.model_dump_json(indent=2))

In [ ]:
# A fixed backend fixture keeps this example independent of a weather-provider key.
# It demonstrates the API boundary; it does not fetch today's weather.
def get_current_weather(location, unit):
    validate({"location": location, "unit": unit}, weather_schema)
    if location.lower() not in {"boston, ma", "boston", "boston, massachusetts"}:
        raise ValueError("The teaching fixture supports Boston only")
    temperature = 22 if unit == "celsius" else 71.6
    return {"location": location, "temperature": temperature, "unit": unit,
            "forecast": ["sunny"]}
tool_call = assistant.tool_calls[0]
assert tool_call.function.name == "get_current_weather"
arguments = json.loads(tool_call.function.arguments)
validate(arguments, weather_schema)
observation = get_current_weather(**arguments)
print(json.dumps(observation, indent=2))

In [ ]:
messages.append(assistant.model_dump(exclude_none=True))
messages.append({"role": "tool", "tool_call_id": tool_call.id,
                 "content": json.dumps(observation)})
final_response = chat(messages, tools=tools, tool_choice="none")
assert final_response.choices[0].message.content
assert not final_response.choices[0].message.tool_calls
assert "22" in final_response.choices[0].message.content
assert "sunny" in final_response.choices[0].message.content.lower()
print(final_response.choices[0].message.content)

In [ ]:
assert observation["temperature"] == 22
assert abs(get_current_weather("Boston, MA", "fahrenheit")["temperature"] - 71.6) < 1e-9
for bad in [{"location": "Boston"}, {"location": "Boston", "unit": "kelvin"},
            {"location": "Boston", "unit": "celsius", "extra": True}]:
    try: validate(bad, weather_schema)
    except ValidationError: pass
    else: raise AssertionError("Invalid arguments accepted")
try: get_current_weather("Paris", "celsius")
except ValueError: pass
else: raise AssertionError("Unsupported location accepted")
assert messages[-1]["tool_call_id"] == messages[-2]["tool_calls"][0]["id"]
print("PASS: schema, temperature conversion, location boundary and tool-call correlation.")

## Original archive appendix — not executed

The original code, prompts and transcripts below are retained verbatim for comparison. Historical SDK calls, placeholders and setup instructions are not the runnable path.

### Original block 1

```text
curl https://api.openai.com/v1/chat/completions -u :$OPENAI_API_KEY -H 'Content-Type: application/json' -d '{
  "model": "gpt-3.5-turbo-0613",
  "messages": [
    {"role": "user", "content": "What is the weather like in Boston?"}
  ],
  "functions": [
    {
      "name": "get_current_weather",
      "description": "Get the current weather in a given location",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "The city and state, e.g. San Francisco, CA"
          },
          "unit": {
            "type": "string",
            "enum": ["celsius", "fahrenheit"]
          }
        },
        "required": ["location"]
      }
    }
  ]
}
```

### Original block 2

```text
{
  "id": "chatcmpl-123",
  ...
  "choices": [{
    "index": 0,
    "message": {
      "role": "assistant",
      "content": null,
      "function_call": {
        "name": "get_current_weather",
        "arguments": "{ \"location\": \"Boston, MA\"}"
      }
    },
    "finish_reason": "function_call"
  }]
}
```

### Original block 3

```text
curl https://weatherapi.com/...
```

### Original block 4

```text
{ "temperature": 22, "unit": "celsius", "description": "Sunny" }
```

### Original block 5

```text
curl https://api.openai.com/v1/chat/completions -u :$OPENAI_API_KEY -H 'Content-Type: application/json' -d '{
  "model": "gpt-3.5-turbo-0613",
  "messages": [
    {"role": "user", "content": "What is the weather like in Boston?"},
    {"role": "assistant", "content": null, "function_call": {"name": "get_current_weather", "arguments": "{ \"location\": \"Boston, MA\"}"}},
    {"role": "function", "name": "get_current_weather", "content": "{\"temperature\": "22", \"unit\": \"celsius\", \"description\": \"Sunny\"}"}
  ],
  "functions": [
    {
      "name": "get_current_weather",
      "description": "Get the current weather in a given location",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "The city and state, e.g. San Francisco, CA"
          },
          "unit": {
            "type": "string",
            "enum": ["celsius", "fahrenheit"]
          }
        },
        "required": ["location"]
      }
    }
  ]
}'
```

### Original block 6

```text
{
  "id": "chatcmpl-123",
  ...
  "choices": [{
    "index": 0,
    "message": {
      "role": "assistant",
      "content": "The weather in Boston is currently sunny with a temperature of 22 degrees Celsius.",
    },
    "finish_reason": "stop"
  }]
}
```

